In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from collections import defaultdict
import scipy.spatial.distance as dist


In [9]:
def cosine_sim(matrix):
    return cosine_similarity(np.nan_to_num(matrix, nan=0.0))

def adjusted_cosine_sim(matrix):
    matrix_centered = matrix - np.nanmean(matrix, axis=1, keepdims=True)
    matrix_centered = np.nan_to_num(matrix_centered, nan=0.0)
    return cosine_similarity(matrix_centered)

def pearson_sim(matrix):
    matrix = np.nan_to_num(matrix, nan=0.0)
    return np.corrcoef(matrix)

def jaccard_sim(matrix):
    binary = ~np.isnan(matrix)
    intersection = np.dot(binary, binary.T)
    union = binary.sum(axis=1)[:, None] + binary.sum(axis=1)[None, :] - intersection
    return intersection / np.maximum(union, 1)

def msd_sim(matrix):
    sim = np.zeros((matrix.shape[0], matrix.shape[0]))
    for i in range(matrix.shape[0]):
        for j in range(i+1, matrix.shape[0]):
            common = ~np.isnan(matrix[i]) & ~np.isnan(matrix[j])
            if np.sum(common) == 0:
                score = 0
            else:
                score = 1 / (1 + np.mean((matrix[i, common] - matrix[j, common]) ** 2))
            sim[i, j] = sim[j, i] = score
    return sim


In [10]:
def predict_ratings(ratings_matrix, similarity_matrix, k=10, user_based=True):
    predictions = np.empty(ratings_matrix.shape)
    for i in range(ratings_matrix.shape[0]):
        for j in range(ratings_matrix.shape[1]):
            if np.isnan(ratings_matrix[i, j]):
                if user_based:
                    sims = similarity_matrix[i]
                    ratings = ratings_matrix[:, j]
                else:
                    sims = similarity_matrix[j]
                    ratings = ratings_matrix[i, :]

                top_k_idx = np.argsort(sims)[-k:]
                top_k_sims = sims[top_k_idx]
                top_k_ratings = ratings[top_k_idx]

                mask = ~np.isnan(top_k_ratings)
                if np.sum(mask) == 0:
                    pred = np.nanmean(ratings_matrix[i]) if user_based else np.nanmean(ratings_matrix[:, j])
                else:
                    pred = np.dot(top_k_sims[mask], top_k_ratings[mask]) / np.sum(np.abs(top_k_sims[mask]))
                predictions[i, j] = pred
            else:
                predictions[i, j] = ratings_matrix[i, j]
    return predictions


In [11]:
def evaluate_accuracy(y_true, y_pred):
    mask = ~np.isnan(y_true)
    mae = mean_absolute_error(y_true[mask], y_pred[mask])
    rmse = root_mean_squared_error(y_true[mask], y_pred[mask])
    return mae, rmse

def evaluate_top_n(y_true, y_pred, threshold=4, top_n=10):
    precision_list, recall_list, f1_list = [], [], []
    for i in range(len(y_true)):
        true_indices = np.where(y_true[i] >= threshold)[0]
        pred_indices = np.argsort(y_pred[i])[-top_n:]

        tp = len(set(true_indices) & set(pred_indices))
        precision = tp / top_n
        recall = tp / len(true_indices) if len(true_indices) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
    return np.mean(precision_list), np.mean(recall_list), np.mean(f1_list)


In [13]:
# Load MovieLens 100k
data = Dataset.load_builtin('ml-100k')
df = pd.DataFrame(data.raw_ratings, columns=["user", "item", "rating", "timestamp"])

n_users = df["user"].nunique()
n_items = df["item"].nunique()

user_map = {u: i for i, u in enumerate(df["user"].unique())}
item_map = {i: j for j, i in enumerate(df["item"].unique())}

ratings_matrix = np.full((n_users, n_items), np.nan)
for row in df.itertuples():
    ratings_matrix[user_map[row.user], item_map[row.item]] = float(row.rating)

# Train/test split (simplified)
mask = ~np.isnan(ratings_matrix)
train_mask = np.random.rand(*ratings_matrix.shape) < 0.8
test_mask = mask & ~train_mask

train_matrix = ratings_matrix.copy()
train_matrix[test_mask] = np.nan

similarity_functions = {
    "Cosine": cosine_sim,
    "AdjCosine": adjusted_cosine_sim,
    "Pearson": pearson_sim,
    "Jaccard": jaccard_sim,
    "MSD": msd_sim
}

results = []

for name, sim_func in similarity_functions.items():
    user_based = True  # or False, depending on the experiment
    sim = sim_func(train_matrix if user_based else train_matrix.T)
    print(f"Similarity matrix ({name}) calculating.")
    for k in [5, 10, 20, 40, 80]:
        print(f"Predicting ratings with k={k} using {name} similarity.")
        pred_matrix = predict_ratings(train_matrix, sim, k=k, user_based=True)

        # Accuracy
        mae, rmse = evaluate_accuracy(ratings_matrix[test_mask], pred_matrix[test_mask])
        # Top-N recommendation metrics
        precision, recall, f1 = evaluate_top_n(ratings_matrix[test_mask], pred_matrix[test_mask])

        results.append({
            "Similarity": name, "K": k,
            "MAE": mae, "RMSE": rmse,
            "Precision": precision, "Recall": recall, "F1": f1
        })

results_df = pd.DataFrame(results)


Similarity matrix (Cosine) calculating.
Predicting ratings with k=5 using Cosine similarity.


KeyboardInterrupt: 